<a href="https://colab.research.google.com/github/eshwar-7419/cads/blob/main/CARC_IDS_Phase2C_v3_Drift_Uncertainty_Retention_Gated_Adaptive_IDS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2C-v3 — Drift–Uncertainty–Retention-Gated Adaptive IDS

This is the next controlled experiment after Phase 2C-v2.

**Problem identified in v2:** all four adaptation candidates were rejected, so v2 prevented forgetting but did not demonstrate continual learning.

**v3 change:** distribution drift alone is no longer treated as proof that retraining is necessary. The controller combines feature drift, score drift, uncertainty, prediction instability, and alert-rate shift. When adaptation is justified, it trains a bounded specialist and fuses it with the protected base IDS:

`p = (1-alpha)*p_base + alpha*p_specialist`

The base model is never replaced.

The candidate is accepted only if protected historical F1/recall degradation and protected benign FPR remain within predefined limits.

Current evaluation labels and final-test labels are never used for controller decisions.


In [1]:
!pip -q install datasets lightgbm psutil joblib scikit-learn scipy

import os, time, json, random, shutil
from pathlib import Path
import numpy as np, pandas as pd, psutil, joblib
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

SEED=42
random.seed(SEED); np.random.seed(SEED)
BASE=Path('/content/carc_ids_phase2c_v3')
RESULTS=BASE/'results'; ARTIFACTS=BASE/'artifacts'
RESULTS.mkdir(parents=True,exist_ok=True); ARTIFACTS.mkdir(parents=True,exist_ok=True)


## 1. Dataset and identical temporal stream

The temporal split and experience definitions are preserved from Phase 2C-v2 so the methodological change is isolated.


In [2]:
ds=load_dataset('lacg030175/UNSW-NB15','temporal')
train_df=ds['train'].to_pandas()
test_df=ds['test'].to_pandas()
y_test=test_df['label'].astype(int).to_numpy()

N_SEGMENTS=10
idxs=np.array_split(np.arange(len(train_df)),N_SEGMENTS)
segments={i:train_df.iloc[idx].copy() for i,idx in enumerate(idxs,1)}
EXPERIENCES={
    'E1_initial_attack':[3],
    'E2_high_attack':[4,5,6],
    'E3_attack_transition':[7],
    'E4_generic_dominant':[8],
    'E5_stable_late':[9,10]
}
experience_dfs={k:pd.concat([segments[i] for i in v],ignore_index=True) for k,v in EXPERIENCES.items()}
display(pd.DataFrame([{'experience':k,'rows':len(v),'benign':int((v.label==0).sum()),'attack':int((v.label==1).sum()),'attack_pct':float(v.label.mean())} for k,v in experience_dfs.items()]))


README.md:   0%|          | 0.00/5.32k [00:00<?, ?B/s]

temporal/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.7MB            

temporal/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

temporal/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.25MB            

temporal/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/175341 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/82332 [00:00<?, ? examples/s]

,experience,rows,benign,attack,attack_pct
0,E1_initial_attack,17534,12842,4692,0.267594
1,E2_high_attack,52602,5405,47197,0.897247
2,E3_attack_transition,17534,2684,14850,0.846926
3,E4_generic_dominant,17534,0,17534,1.000000
4,E5_stable_late,35068,0,35068,1.000000


## 2. Leakage-safe preprocessing and E1 protocol

E1 is split into 56% train, 7% calibration, 7% protected validation, and 30% attack evaluation.

Protected validation is never used for training/replay.


In [3]:
DROP=[c for c in ['label','attack_cat','id','ID','index'] if c in train_df.columns]
Xraw=train_df.drop(columns=DROP,errors='ignore')
Xtestraw=test_df.drop(columns=DROP,errors='ignore')
num=Xraw.select_dtypes(include=[np.number]).columns.tolist()
cat=[c for c in Xraw.columns if c not in num]
prep=ColumnTransformer([
 ('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),
 ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)
])
prep.fit(Xraw)
joblib.dump(prep,ARTIFACTS/'preprocessor.joblib')

def tr(df):
    return prep.transform(df.drop(columns=DROP,errors='ignore')).astype(np.float32),df.label.astype(int).to_numpy()

e1=experience_dfs['E1_initial_attack']
e1tr,rest=train_test_split(e1,test_size=.44,stratify=e1.label,random_state=SEED)
cal,rest2=train_test_split(rest,test_size=37/44,stratify=rest.label,random_state=SEED)
prot,attackeval=train_test_split(rest2,test_size=30/37,stratify=rest2.label,random_state=SEED)
E1_X,E1_y=tr(e1tr); CAL_X,CAL_y=tr(cal); PROT_X,PROT_y=tr(prot); E1E_X,E1E_y=tr(attackeval)

ben=pd.concat([segments[1],segments[2]],ignore_index=True)
ben=ben[ben.label==0].sample(n=min(10000,(ben.label==0).sum()),random_state=SEED)
BEN_X,BEN_y=tr(ben)

XTEST=prep.transform(Xtestraw).astype(np.float32)

stream={'E1_initial_attack':{'X_adapt':E1_X,'y_adapt':E1_y,'X_attack_eval':E1E_X,'y_attack_eval':E1E_y}}
for name in list(EXPERIENCES)[1:]:
    d=experience_dfs[name]; a=d[d.label==1].copy(); s=max(1,min(int(.70*len(a)),len(a)-1))
    ad,ev=a.iloc[:s],a.iloc[s:]
    stream[name]={'X_adapt':tr(ad)[0],'y_adapt':tr(ad)[1],'X_attack_eval':tr(ev)[0],'y_attack_eval':tr(ev)[1]}

def operational(data):
    n=min(len(BEN_X),len(data['X_attack_eval']))
    return np.r_[BEN_X[:n],data['X_attack_eval']],np.r_[BEN_y[:n],data['y_attack_eval']]
for d in stream.values():
    d['X_eval'],d['y_eval']=operational(d)


## 3. Metrics, base model, protected operating point


In [4]:
def model(n=300):
    return LGBMClassifier(objective='binary',n_estimators=n,learning_rate=.05,num_leaves=31,random_state=SEED,n_jobs=-1,verbosity=-1)

def metrics(y,p,t):
    z=(p>=t).astype(int); tn,fp,fn,tp=confusion_matrix(y,z,labels=[0,1]).ravel()
    return {'accuracy':accuracy_score(y,z),'precision':precision_score(y,z,zero_division=0),'recall':recall_score(y,z,zero_division=0),'f1':f1_score(y,z,zero_division=0),'fpr':fp/(fp+tn) if fp+tn else np.nan,'specificity':tn/(tn+fp) if tn+fp else np.nan,'balanced_accuracy':((tp/(tp+fn) if tp+fn else 0)+(tn/(tn+fp) if tn+fp else 0))/2,'roc_auc':roc_auc_score(y,p),'pr_auc':average_precision_score(y,p),'tp':tp,'fp':fp,'tn':tn,'fn':fn}

def threshold(y,p,limit=.10):
    rows=[]
    for t in np.linspace(.05,.95,181):
        m=metrics(y,p,t); rows.append((t,m['f1'],m['recall'],m['fpr']))
    q=pd.DataFrame(rows,columns=['threshold','f1','recall','fpr'])
    f=q[q.fpr<=limit]
    r=(f if len(f) else q).sort_values(['f1','recall','fpr'],ascending=[False,False,True]).iloc[0]
    return float(r.threshold)

base=model()
t0=time.perf_counter(); base.fit(E1_X,E1_y); initial_time=time.perf_counter()-t0
base_threshold=threshold(CAL_y,base.predict_proba(CAL_X)[:,1],.10)

prot_p=base.predict_proba(PROT_X)[:,1]
ben_p=base.predict_proba(BEN_X)[:,1]
protected_base=metrics(PROT_y,prot_p,base_threshold)
protected_base['benign_fpr']=float(np.mean(ben_p>=base_threshold))
display(pd.Series(protected_base).round(6))


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,0
accuracy,0.923390
precision,0.846154
recall,0.871951
f1,0.858859
fpr,0.057842
specificity,0.942158
balanced_accuracy,0.907055
roc_auc,0.979445
pr_auc,0.954740
tp,286.000000


## 4. Drift, score drift, uncertainty and instability

The controller uses only deployment-observable signals.

- PSI for feature drift
- PSI for model-score drift
- binary entropy for uncertainty
- small perturbation test for prediction instability
- alert-rate shift


In [5]:
def psi1(a,b,bins=10):
    edges=np.unique(np.quantile(a,np.linspace(0,1,bins+1)))
    if len(edges)<3:return 0.
    x,_=np.histogram(a,bins=edges); y,_=np.histogram(b,bins=edges); e=1e-6
    x=(x+e)/(x.sum()+e*len(x)); y=(y+e)/(y.sum()+e*len(y))
    return float(np.sum((x-y)*np.log(x/y)))

def feature_drift(ref,x,maxf=150):
    d=min(ref.shape[1],x.shape[1],maxf)
    vals=[psi1(ref[:,j],x[:,j]) for j in range(d)]
    return {'p90_psi':float(np.quantile(vals,.90)),'median_psi':float(np.median(vals)),'max_psi':float(np.max(vals))}

def score_drift(a,b): return psi1(a,b)

def entropy(p):
    p=np.clip(p,1e-7,1-1e-7)
    return -(p*np.log2(p)+(1-p)*np.log2(1-p))

def uncertainty(p):
    h=entropy(p)
    return {'mean_entropy':float(h.mean()),'high_uncertainty_fraction':float(np.mean(h>=.80))}

def instability(m,x,n=3000,scale=.02):
    rng=np.random.default_rng(SEED); n=min(n,len(x)); ix=rng.choice(len(x),n,replace=False)
    xx=x[ix]; p=m.predict_proba(xx)[:,1]
    xp=(xx+rng.normal(0,scale,xx.shape)).astype(np.float32)
    q=m.predict_proba(xp)[:,1]
    return float(np.mean((p>=.5)!=(q>=.5)))

MOD_PSI=.10; STRONG_PSI=.20; MOD_SCORE=.10; STRONG_SCORE=.20
HIGH_UNCERT=.20; HIGH_INST=.10; MOD_ALERT=.10; STRONG_ALERT=.20
MAX_F1_DROP=.05; MAX_RECALL_DROP=.05; MAX_PROT_FPR=.10
ALPHA_MIN=.10; ALPHA_MAX=.35
LIGHT_MEM=1000; REPLAY_MEM=8000


## 5. Adaptation-necessity gate

Feature drift alone is insufficient.

The controller requests adaptation only when drift is accompanied by evidence that the decision function is under stress.


In [6]:
def necessity(fd,sd,u,inst,ashift):
    strong_f=fd['p90_psi']>=STRONG_PSI; mod_f=fd['p90_psi']>=MOD_PSI
    strong_s=sd>=STRONG_SCORE; mod_s=sd>=MOD_SCORE
    hu=u['high_uncertainty_fraction']>=HIGH_UNCERT
    hi=inst>=HIGH_INST
    sa=ashift>=STRONG_ALERT; ma=ashift>=MOD_ALERT
    if strong_f and (strong_s or hu or hi): return 'STRONG_ADAPT'
    if sa and (hu or hi): return 'STRONG_ADAPT'
    if (mod_f or mod_s) and (hu or hi or ma): return 'LIGHT_ADAPT'
    return 'NO_UPDATE'

def alpha_value(fd,u,inst):
    s=np.mean([np.clip(fd['p90_psi'],0,1),np.clip(u['high_uncertainty_fraction'],0,1),np.clip(inst,0,1)])
    return float(np.clip(ALPHA_MIN+s*(ALPHA_MAX-ALPHA_MIN),ALPHA_MIN,ALPHA_MAX))


## 6. Bounded specialist and retention gate

The base IDS remains intact. A specialist is only an additive bounded component.

A rejected specialist never enters replay memory.


In [7]:
def replay_update(mx,my,cx,cy,budget):
    X=np.r_[mx,cx]; y=np.r_[my,cy]
    rng=np.random.default_rng(SEED); cls=np.unique(y)
    if len(cls)<2:
        ix=rng.choice(len(y),min(budget,len(y)),replace=False); return X[ix],y[ix]
    chosen=[]
    for c in cls:
        ix=np.where(y==c)[0]; chosen.append(rng.choice(ix,min(len(ix),max(1,budget//len(cls))),replace=False))
    ix=np.concatenate(chosen)
    if len(ix)>budget: ix=rng.choice(ix,budget,replace=False)
    return X[ix],y[ix]

def fit_specialist(cx,cy,mx,my,n=80):
    X=np.r_[cx,mx]; y=np.r_[cy,my]
    sp=LGBMClassifier(objective='binary',n_estimators=n,learning_rate=.05,num_leaves=15,max_depth=7,random_state=SEED,n_jobs=-1,verbosity=-1)
    before=psutil.Process(os.getpid()).memory_info().rss/(1024**2); t=time.perf_counter(); sp.fit(X,y); elapsed=time.perf_counter()-t
    after=psutil.Process(os.getpid()).memory_info().rss/(1024**2)
    return sp,elapsed,max(before,after)

def fused(base_model,specialist,alpha,X):
    p=base_model.predict_proba(X)[:,1]
    if specialist is None:return p
    q=specialist.predict_proba(X)[:,1]
    return (1-alpha)*p+alpha*q

def gate(base_model,specialist,alpha):
    p=fused(base_model,specialist,alpha,PROT_X)
    b=fused(base_model,specialist,alpha,BEN_X)
    m=metrics(PROT_y,p,base_threshold)
    bfpr=float(np.mean(b>=base_threshold))
    ok=(m['f1']>=protected_base['f1']-MAX_F1_DROP and m['recall']>=protected_base['recall']-MAX_RECALL_DROP and bfpr<=MAX_PROT_FPR)
    return ok,m['f1'],m['recall'],bfpr


## 7. Main v3 controller

The current adaptation labels are accessed only after the label-free necessity gate requests a specialist.

The final evaluation labels are never used in the controller.


In [8]:
accepted_specialist=None; accepted_alpha=0.
memory_X,memory_y=replay_update(E1_X,E1_y,np.empty((0,E1_X.shape[1]),dtype=np.float32),np.empty(0,dtype=int),min(REPLAY_MEM,len(E1_y)))
reference_X=E1_X.copy()
previous_scores=base.predict_proba(E1_X)[:,1]
previous_alert=float(np.mean(previous_scores>=base_threshold))
states={'E1_initial_attack':(accepted_specialist,accepted_alpha)}
rows=[]

for name in list(EXPERIENCES)[1:]:
    d=stream[name]
    current=fused(base,accepted_specialist,accepted_alpha,d['X_adapt'])
    fd=feature_drift(reference_X,d['X_adapt'])
    sd=score_drift(previous_scores,current)
    u=uncertainty(current)
    inst=instability(base,d['X_adapt'])
    ar=float(np.mean(current>=base_threshold)); ashift=abs(ar-previous_alert)
    need=necessity(fd,sd,u,inst,ashift)

    specialist=None; alpha=0.; elapsed=0.; rss=0.; accepted=False; rollback=False; action='NO_UPDATE'
    cf1=protected_base['f1']; crec=protected_base['recall']; cfpr=protected_base['benign_fpr']

    if need in ['LIGHT_ADAPT','STRONG_ADAPT']:
        action='LIGHT_SPECIALIST' if need=='LIGHT_ADAPT' else 'REPLAY_SPECIALIST'
        mem_budget=LIGHT_MEM if need=='LIGHT_ADAPT' else REPLAY_MEM
        est=80 if need=='LIGHT_ADAPT' else 180
        mx,my=replay_update(memory_X,memory_y,d['X_adapt'],d['y_adapt'],mem_budget)
        specialist,elapsed,rss=fit_specialist(d['X_adapt'],d['y_adapt'],mx,my,est)
        alpha=alpha_value(fd,u,inst)
        accepted,cf1,crec,cfpr=gate(base,specialist,alpha)
        if accepted:
            accepted_specialist=specialist; accepted_alpha=alpha
            memory_X,memory_y=replay_update(memory_X,memory_y,d['X_adapt'],d['y_adapt'],REPLAY_MEM)
            reference_X=memory_X.copy()
        else:
            rollback=True

    states[name]=(accepted_specialist,accepted_alpha)
    rows.append({
        'experience':name,'adaptation_decision':need,'candidate_action':action,
        'accepted':accepted,'rollback':rollback,'feature_p90_psi':fd['p90_psi'],
        'score_psi':sd,'mean_entropy':u['mean_entropy'],
        'high_uncertainty_fraction':u['high_uncertainty_fraction'],
        'instability':inst,'alert_rate':ar,'alert_rate_shift':ashift,
        'alpha':accepted_alpha,'candidate_time_sec':elapsed,'candidate_rss_mb':rss,
        'replay_memory_samples':len(memory_y),'candidate_protected_f1':cf1,
        'candidate_protected_recall':crec,'candidate_protected_fpr':cfpr
    })
    previous_scores=current; previous_alert=ar

decisions=pd.DataFrame(rows)
display(decisions.round(4))
decisions.to_csv(RESULTS/'controller_v3_decisions.csv',index=False)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,experience,adaptation_decision,candidate_action,accepted,rollback,feature_p90_psi,score_psi,mean_entropy,high_uncertainty_fraction,instability,alert_rate,alert_rate_shift,alpha,candidate_time_sec,candidate_rss_mb,replay_memory_samples,candidate_protected_f1,candidate_protected_recall,candidate_protected_fpr
0,E2_high_attack,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,0.3169,7.8660,0.2821,0.1360,0.1937,0.8418,0.5626,0.1539,7.8018,1100.0000,8000,0.8563,0.9085,0.0120
1,E3_attack_transition,LIGHT_ADAPT,LIGHT_SPECIALIST,True,False,0.1665,0.1520,0.2913,0.1564,0.1693,0.9018,0.0599,0.1410,0.2988,1085.9023,8000,0.8563,0.9085,0.0113
2,E4_generic_dominant,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,1.0227,0.5495,0.4827,0.2886,0.2513,0.8642,0.0376,0.2283,0.8355,1085.9219,8000,0.8571,0.8872,0.0109
3,E5_stable_late,STRONG_ADAPT,REPLAY_SPECIALIST,True,False,0.8238,0.1243,0.4860,0.3228,0.2327,0.9014,0.0372,0.2149,0.8819,1106.1094,8000,0.8588,0.8994,0.0106


## 8. Current-experience performance


In [9]:
cur=[]
for name,(sp,a) in states.items():
    d=stream[name]; p=fused(base,sp,a,d['X_eval']); m=metrics(d['y_eval'],p,base_threshold)
    cur.append({'model_state':name,'evaluated_experience':name,'alpha':a,**m})
current_results=pd.DataFrame(cur)
display(current_results[['model_state','f1','recall','precision','fpr','balanced_accuracy','alpha']].round(4))
current_results.to_csv(RESULTS/'controller_v3_current_results.csv',index=False)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,model_state,f1,recall,precision,fpr,balanced_accuracy,alpha
0,E1_initial_attack,0.8559,0.8899,0.8243,0.0293,0.9303,0.0000
1,E2_high_attack,0.9326,0.8811,0.9905,0.0120,0.9346,0.1539
2,E3_attack_transition,0.9321,0.8819,0.9884,0.0103,0.9358,0.1410
3,E4_generic_dominant,0.9372,0.8917,0.9876,0.0112,0.9402,0.2283
4,E5_stable_late,0.9442,0.9033,0.9890,0.0106,0.9464,0.2149


## 9. Final retention and independent temporal test


In [10]:
final_sp,final_a=states['E5_stable_late']
fr=[]
for name,d in stream.items():
    p=fused(base,final_sp,final_a,d['X_eval']); m=metrics(d['y_eval'],p,base_threshold)
    fr.append({'evaluated_experience':name,'alpha':final_a,**m})
final_retention=pd.DataFrame(fr)
display(final_retention[['evaluated_experience','f1','recall','precision','fpr','balanced_accuracy']].round(4))
final_retention.to_csv(RESULTS/'controller_v3_final_retention.csv',index=False)

test_p=fused(base,final_sp,final_a,XTEST)
test_m=metrics(y_test,test_p,base_threshold)
final_test=pd.DataFrame([{'method':'Drift_Uncertainty_Retention_Controller_v3','threshold':base_threshold,'alpha':final_a,**test_m}])
display(final_test.round(6))
final_test.to_csv(RESULTS/'controller_v3_final_temporal_test.csv',index=False)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

,evaluated_experience,f1,recall,precision,fpr,balanced_accuracy
0,E1_initial_attack,0.8436,0.9062,0.7891,0.0374,0.9344
1,E2_high_attack,0.9329,0.8807,0.9916,0.0106,0.9351
2,E3_attack_transition,0.9509,0.9152,0.9896,0.0097,0.9527
3,E4_generic_dominant,0.9371,0.8913,0.9878,0.0110,0.9401
4,E5_stable_late,0.9442,0.9033,0.9890,0.0106,0.9464


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,method,threshold,alpha,accuracy,precision,recall,f1,fpr,specificity,balanced_accuracy,roc_auc,pr_auc,tp,fp,tn,fn
0,Drift_Uncertainty_Retention_Controller_v3,0.34,0.214936,0.876986,0.879943,0.899276,0.889505,0.150324,0.849676,0.874476,0.94783,0.953165,40766,5562,31438,4566


## 10. Forgetting, efficiency and resource decision analysis


In [11]:
forget=[]
for name in stream:
    i=float(current_results.loc[current_results.model_state==name,'f1'].iloc[0])
    f=float(final_retention.loc[final_retention.evaluated_experience==name,'f1'].iloc[0])
    forget.append({'experience':name,'initial_f1':i,'final_f1':f,'forgetting':max(0,i-f)})
forgetting=pd.DataFrame(forget); display(forgetting.round(6)); forgetting.to_csv(RESULTS/'controller_v3_forgetting.csv',index=False)

later=decisions
eff=pd.DataFrame([{
    'method':'Drift_Uncertainty_Retention_Controller_v3',
    'candidate_compute_sec':float(later.candidate_time_sec.sum()),
    'accepted_adaptations':int(later.accepted.sum()),
    'rollbacks':int(later.rollback.sum()),
    'light_specialist_requests':int((later.candidate_action=='LIGHT_SPECIALIST').sum()),
    'replay_specialist_requests':int((later.candidate_action=='REPLAY_SPECIALIST').sum()),
    'no_update_requests':int((later.candidate_action=='NO_UPDATE').sum()),
    'max_replay_memory_samples':int(later.replay_memory_samples.max()),
    'max_candidate_rss_mb':float(later.candidate_rss_mb.max())
}])
display(eff.round(4)); eff.to_csv(RESULTS/'controller_v3_efficiency.csv',index=False)

resource=[]
for scen,spec in {'generous':(60,12000),'moderate':(20,4000),'constrained':(10,1000)}.items():
    for _,r in decisions.iterrows():
        act=r.candidate_action
        if act=='REPLAY_SPECIALIST' and spec[1]<REPLAY_MEM: act='LIGHT_SPECIALIST'
        if act=='LIGHT_SPECIALIST' and spec[1]<LIGHT_MEM: act='NO_UPDATE'
        resource.append({'scenario':scen,'experience':r.experience,'controller_decision':r.adaptation_decision,'resource_feasible_action':act,'time_budget_sec':spec[0],'memory_budget_samples':spec[1]})
resource_df=pd.DataFrame(resource); display(resource_df); resource_df.to_csv(RESULTS/'controller_v3_resource_scenarios.csv',index=False)


,experience,initial_f1,final_f1,forgetting
0,E1_initial_attack,0.855874,0.843636,0.012238
1,E2_high_attack,0.932616,0.932865,0.000000
2,E3_attack_transition,0.932147,0.950904,0.000000
3,E4_generic_dominant,0.937169,0.937050,0.000119
4,E5_stable_late,0.944215,0.944215,0.000000


,method,candidate_compute_sec,accepted_adaptations,rollbacks,light_specialist_requests,replay_specialist_requests,no_update_requests,max_replay_memory_samples,max_candidate_rss_mb
0,Drift_Uncertainty_Retention_Controller_v3,9.8179,4,0,1,3,0,8000,1106.1094


,scenario,experience,controller_decision,resource_feasible_action,time_budget_sec,memory_budget_samples
0,generous,E2_high_attack,STRONG_ADAPT,REPLAY_SPECIALIST,60,12000
1,generous,E3_attack_transition,LIGHT_ADAPT,LIGHT_SPECIALIST,60,12000
2,generous,E4_generic_dominant,STRONG_ADAPT,REPLAY_SPECIALIST,60,12000
3,generous,E5_stable_late,STRONG_ADAPT,REPLAY_SPECIALIST,60,12000
4,moderate,E2_high_attack,STRONG_ADAPT,LIGHT_SPECIALIST,20,4000
5,moderate,E3_attack_transition,LIGHT_ADAPT,LIGHT_SPECIALIST,20,4000
6,moderate,E4_generic_dominant,STRONG_ADAPT,LIGHT_SPECIALIST,20,4000
7,moderate,E5_stable_late,STRONG_ADAPT,LIGHT_SPECIALIST,20,4000
8,constrained,E2_high_attack,STRONG_ADAPT,LIGHT_SPECIALIST,10,1000
9,constrained,E3_attack_transition,LIGHT_ADAPT,LIGHT_SPECIALIST,10,1000


## 11. Optional Phase-2B comparison

If `final_temporal_test_comparison.csv` is uploaded to `/content`, it is appended without changing its values. Missing baseline files are never fabricated.


In [14]:
cands=[Path('/content/final_temporal_test_comparison.csv'),Path('/content/final_temporal_test_comparison (2).csv')]
bdf=None
for p in cands:
    if p.exists():
        try:
            z=pd.read_csv(p)
            if 'method' in z.columns:bdf=z;break
        except: pass
if bdf is not None:
    cols=[c for c in ['method','threshold','f1','recall','precision','fpr','balanced_accuracy','roc_auc','pr_auc'] if c in bdf.columns and c in final_test.columns]
    combined=pd.concat([bdf[cols],final_test[cols]],ignore_index=True)
    display(combined.round(6))
    combined.to_csv(RESULTS/'phase2c_v3_final_comparison.csv',index=False)
else:
    print('Phase-2B comparison CSV not found; upload it later for the combined table.')


,method,f1,recall,precision,fpr,roc_auc,pr_auc
0,Static_LightGBM,0.000000,0.000000,0.000000,0.000000,0.500000,0.550600
1,Naive_CL_LightGBM,0.000000,0.000000,0.000000,0.000000,0.500000,0.550600
2,Replay_LightGBM,0.881700,0.982110,0.799917,0.300973,0.976623,0.982514
3,EWC_MLP,0.715016,1.000000,0.556439,0.976649,0.694992,0.668194
4,Drift_Uncertainty_Retention_Controller_v3,0.889505,0.899276,0.879943,0.150324,0.947830,0.953165


## 12. Save protocol and artifacts

Interpretation rule:

- Do not loosen the retention gate merely to manufacture accepted updates.
- If v3 still accepts zero updates, that is evidence that this dataset/model combination does not justify continual adaptation under the stated constraints.
- If accepted specialists improve later regimes while preserving E1, the adaptive IDS claim becomes substantially stronger.


In [15]:
protocol={
 'phase':'2C-v3',
 'name':'Drift-Uncertainty-Retention-Gated Adaptive IDS',
 'dataset':'lacg030175/UNSW-NB15',
 'configuration':'temporal',
 'experiences':EXPERIENCES,
 'signals':['feature_PSI','score_PSI','entropy_uncertainty','perturbation_instability','alert_rate_shift'],
 'actions':['NO_UPDATE','LIGHT_SPECIALIST','REPLAY_SPECIALIST'],
 'alpha_range':[ALPHA_MIN,ALPHA_MAX],
 'retention':{'max_f1_drop':MAX_F1_DROP,'max_recall_drop':MAX_RECALL_DROP,'max_protected_fpr':MAX_PROT_FPR},
 'controller_uses_current_evaluation_labels':False,
 'controller_uses_final_test_labels':False,
 'candidate_training_uses_current_labels_after_gate':True,
 'seed':SEED
}
with open(RESULTS/'phase2c_v3_protocol.json','w') as f: json.dump(protocol,f,indent=2)
joblib.dump(base,ARTIFACTS/'v3_base_model.joblib')
if final_sp is not None: joblib.dump(final_sp,ARTIFACTS/'v3_final_specialist.joblib')
zip_path=shutil.make_archive(str(BASE/'phase2c_v3_artifacts'),'zip',root_dir=BASE)
print('Created:',zip_path)
print('\nResults:')
for p in sorted(RESULTS.glob('*')): print(p)


Created: /content/carc_ids_phase2c_v3/phase2c_v3_artifacts.zip

Results:
/content/carc_ids_phase2c_v3/results/controller_v3_current_results.csv
/content/carc_ids_phase2c_v3/results/controller_v3_decisions.csv
/content/carc_ids_phase2c_v3/results/controller_v3_efficiency.csv
/content/carc_ids_phase2c_v3/results/controller_v3_final_retention.csv
/content/carc_ids_phase2c_v3/results/controller_v3_final_temporal_test.csv
/content/carc_ids_phase2c_v3/results/controller_v3_forgetting.csv
/content/carc_ids_phase2c_v3/results/controller_v3_resource_scenarios.csv
/content/carc_ids_phase2c_v3/results/phase2c_v3_final_comparison.csv
/content/carc_ids_phase2c_v3/results/phase2c_v3_protocol.json
